# 02 — Diagnóstico de Esquemas y Preparación de Reconstrucción

Este notebook corresponde a la **sección de Persona 1** del diagnóstico. Compara esquemas reales contra esquemas esperados y deja preparados los JSON de metadatos para Persona 2.

In [1]:
!pip install -q pyspark pyyaml
!pip install -q pyarrow



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import sys
import json
from pathlib import Path

# Ruta del proyecto
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Agregar carpeta src al path
sys.path.append(str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)

# Configuración necesaria para PySpark en Windows
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

print("Proyecto:", PROJECT_ROOT)
print("Python usado por Spark:", sys.executable)
print("HADOOP_HOME:", os.environ["HADOOP_HOME"])

Proyecto: C:\Users\HP OMEN\Modelado\SegundoParcial\Proyecto_Grupo1
Python usado por Spark: C:\Program Files\Python314\python.exe
HADOOP_HOME: C:\hadoop


In [3]:
import importlib
import schema_recovery
importlib.reload(schema_recovery)
from schema_recovery import diagnose_successful_files, write_schema_differences
from utils import load_config, get_spark_session

config = load_config("config/etl_config.yaml")
spark = get_spark_session(config)
spark.sparkContext.setLogLevel("WARN")


## 1. Cargar inventario técnico generado 

In [4]:
import pandas as pd
from pyspark.sql import functions as F

audit_path = Path(config["paths"]["audit"]) / "audit_file_inventory"
inventory_pd = pd.read_parquet(str(audit_path), engine="pyarrow")

inventory_str = inventory_pd.astype(str).replace(
    {"nan": None, "<NA>": None, "NaT": None, "None": None, "": None}
)

inventory_df = spark.createDataFrame(inventory_str)

# try_cast devuelve NULL en vez de error para NaN/valores inválidos
inventory_df = (inventory_df
    .withColumn("file_size_mb",    F.expr("try_cast(file_size_mb as double)"))
    .withColumn("partition_year",  F.expr("try_cast(try_cast(partition_year as double) as bigint)"))
    .withColumn("partition_month", F.expr("try_cast(try_cast(partition_month as double) as bigint)"))
    .withColumn("record_count",    F.expr("try_cast(try_cast(record_count as double) as bigint)"))
    .withColumn("column_count",    F.expr("try_cast(try_cast(column_count as double) as bigint)"))
)

inventory_df.show(30, truncate=80)


+------------------------------------+----------------------+------------+-----------------------------------+--------------------------------------------------------------------------------+------------+----------------------------------------------------------------+--------------+---------------+------------+------------+------------+----------------------------------------------------------------+--------------------------------------------------------------------------------+--------------------------+
|                          process_id|         source_system|service_type|                          file_name|                                                                       file_path|file_size_mb|                                                file_hash_sha256|partition_year|partition_month| read_status|record_count|column_count|                                                     schema_hash|                                                                   error_message|  

## 2. Mostrar esquemas esperados creados en metadata/

In [5]:
metadata_files = [
    "metadata/expected_schema_yellow.json",
    "metadata/expected_schema_green.json",
    "metadata/expected_schema_fhvhv.json",
    "metadata/canonical_schema_trips.json",
    "metadata/business_rules.json",
]

for file in metadata_files:
    print("\n===", file, "===")
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(json.dumps(data, indent=2, ensure_ascii=False)[:1500])


=== metadata/expected_schema_yellow.json ===
{
  "service_type": "yellow",
  "description": "Esquema esperado aproximado para archivos NYC TLC Yellow Taxi Trip Data.",
  "fields": [
    {
      "name": "VendorID",
      "type": "long",
      "nullable": true
    },
    {
      "name": "tpep_pickup_datetime",
      "type": "timestamp",
      "nullable": true
    },
    {
      "name": "tpep_dropoff_datetime",
      "type": "timestamp",
      "nullable": true
    },
    {
      "name": "passenger_count",
      "type": "double",
      "nullable": true
    },
    {
      "name": "trip_distance",
      "type": "double",
      "nullable": true
    },
    {
      "name": "RatecodeID",
      "type": "double",
      "nullable": true
    },
    {
      "name": "store_and_fwd_flag",
      "type": "string",
      "nullable": true
    },
    {
      "name": "PULocationID",
      "type": "integer",
      "nullable": true
    },
    {
      "name": "DOLocationID",
      "type": "integer",
      "nul

## 3. Comparación de esquema real vs esquema esperado

Se generan tres tipos de diferencia: `EXTRA_COLUMN`, `MISSING_COLUMN` y `TYPE_MISMATCH`.

In [6]:
df_check = inventory_df.toPandas()
print("Total filas:", len(df_check))
print("Valores read_status:", df_check["read_status"].value_counts().to_dict())


Total filas: 20
Valores read_status: {'SUCCESS': 18, 'SCHEMA_ERROR': 1, 'CORRUPT': 1}


In [7]:
df_inv = inventory_df.toPandas()
result = (df_inv[df_inv["read_status"] == "SUCCESS"]
    .groupby(["service_type", "schema_hash"])
    .size()
    .reset_index(name="count")
    .sort_values(["service_type", "count"]))
print(result.to_string(index=False))


service_type                                                      schema_hash  count
 bad_parquet 145ef2eda729c78f944e6b1bfe4fb8a076f2222eee59c616f996e38e06fd726a      1
 bad_parquet 2ee393a89b3ae9815f3e24f047f03d2a0fcf69d059e4bd8930290830c43de52c      1
 bad_parquet 7e3103878695cb996098b571752d1ebf746ae6c1366189cfbbc50eaa9acc2798      1
 bad_parquet 89ae3cf20730e6fa8d84afe41367821a61036f5bbd280705834456960a506253      1
 bad_parquet f98ab85e87dd0b98219c10b071b90b17e318fe6bd3aa1268e055d8dcf8331263      1
 bad_parquet 92681c0c3ddf2795b6c197dcddf2b9a13a51f824d0ba378d753ae18e969d0a66      2
       fhvhv db4cd5c4701313fb72a9082181dc2497a3df672bd4820d94e544377d0440252e      1
       green 003d7291ad2ae7399c90f5da09fed57bf9493d755058ef9ab946e8ef7317b823      1
       green 3a7c6d677e6b981a890fd8799a839b2ad128093013f3e4abf2d714f09e92ec94      1
      yellow 9c74eb92d675451104d7898d717cc313b0e0150024da7a1c4dd88034a72b4d8e      1
      yellow f8ace373f12c24ff3bdccfa9da15cb3415082626673c98d1aa01

In [8]:
import importlib
import schema_recovery
importlib.reload(schema_recovery)
from schema_recovery import diagnose_successful_files, write_schema_differences

# Calcula schema_diff_df si no está definido (ejecución de celda individual)
if "schema_diff_df" not in globals():
    process_id = inventory_df.select("process_id").first()[0]
    schema_diff_df = diagnose_successful_files(spark, config, inventory_df, process_id)
    print(f"schema_diff_df calculado: {schema_diff_df.count()} diferencias")

schema_diff_path = write_schema_differences(schema_diff_df, config)
print("schema_differences escrito en:", schema_diff_path)


schema_diff_df calculado: 36 diferencias
schema_differences escrito en: C:\Users\HP OMEN\Modelado\SegundoParcial\Proyecto_Grupo1\data\audit\schema_differences


## 4. Resumen de diferencias por servicio

In [9]:
# Guard: recalcular schema_diff_df si no está en contexto
if "schema_diff_df" not in globals():
    import importlib, schema_recovery
    importlib.reload(schema_recovery)
    from schema_recovery import diagnose_successful_files
    process_id = inventory_df.select("process_id").first()[0]
    schema_diff_df = diagnose_successful_files(spark, config, inventory_df, process_id)

df_counts = (schema_diff_df.toPandas()
    .groupby(["service_type", "diff_type"])
    .size()
    .reset_index(name="count")
    .sort_values(["service_type", "diff_type"]))
print(df_counts.to_string(index=False))


service_type      diff_type  count
       fhvhv  TYPE_MISMATCH      2
       green  TYPE_MISMATCH      8
      yellow   EXTRA_COLUMN      3
      yellow MISSING_COLUMN      3
      yellow  TYPE_MISMATCH     20


## 5. Hashes de esquema por servicio

Permite identificar cambios estructurales entre meses o versiones del mismo tipo de servicio.

In [10]:
inventory_df.filter("read_status = 'SUCCESS'").groupBy(
    "service_type", "schema_hash"
).count().orderBy("service_type", "count").show(100, truncate=False)

+------------+----------------------------------------------------------------+-----+
|service_type|schema_hash                                                     |count|
+------------+----------------------------------------------------------------+-----+
|bad_parquet |89ae3cf20730e6fa8d84afe41367821a61036f5bbd280705834456960a506253|1    |
|bad_parquet |f98ab85e87dd0b98219c10b071b90b17e318fe6bd3aa1268e055d8dcf8331263|1    |
|bad_parquet |145ef2eda729c78f944e6b1bfe4fb8a076f2222eee59c616f996e38e06fd726a|1    |
|bad_parquet |7e3103878695cb996098b571752d1ebf746ae6c1366189cfbbc50eaa9acc2798|1    |
|bad_parquet |2ee393a89b3ae9815f3e24f047f03d2a0fcf69d059e4bd8930290830c43de52c|1    |
|bad_parquet |92681c0c3ddf2795b6c197dcddf2b9a13a51f824d0ba378d753ae18e969d0a66|2    |
|fhvhv       |db4cd5c4701313fb72a9082181dc2497a3df672bd4820d94e544377d0440252e|1    |
|green       |003d7291ad2ae7399c90f5da09fed57bf9493d755058ef9ab946e8ef7317b823|1    |
|green       |3a7c6d677e6b981a890fd8799a839b2ad1280930

---

## Sección 2 — Reconstrucción del Esquema Canónico 

Esta sección toma los DataFrames exitosamente leídos y los homologa al **esquema canónico de 24 campos** definido en `canonical_schema_trips.json`, usando las reglas de `business_rules.json`.

Pasos:
1. Importar funciones 
2. Aplicar `apply_canonical_schema()` a cada archivo SUCCESS por servicio
3. Clasificar archivos dañados y generar registros de quarantine
4. Unir los DataFrames canónicos y escribir en `data/bronze/`

In [11]:
from schema_recovery import (
    apply_canonical_schema,
    apply_canonical_schema_pd,
    classify_corrupt_file,
    generate_quarantine_record,
    CANONICAL_FIELDS,
)

print("Funciones importadas correctamente.")
print("Campos canónicos:", CANONICAL_FIELDS)

Funciones importadas correctamente.
Campos canónicos: ['trip_id', 'service_type', 'vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'pickup_location_id', 'dropoff_location_id', 'payment_type', 'fare_amount', 'extra_amount', 'mta_tax', 'tip_amount', 'tolls_amount', 'total_amount', 'congestion_surcharge', 'airport_fee', 'year', 'month', 'source_file', 'ingestion_timestamp', 'improvement_surcharge', 'quality_status']


In [12]:
import pandas as pd
import gc
import shutil
from pathlib import Path

# ── Preparar directorio bronze ────────────────────────────────────────────────
bronze_path_str = str(PROJECT_ROOT / config["paths"]["bronze"])
bronze_path_dir = Path(bronze_path_str)
if bronze_path_dir.exists():
    shutil.rmtree(str(bronze_path_dir))
bronze_path_dir.mkdir(parents=True, exist_ok=True)
print(f"Bronze output: {bronze_path_dir}\n")

# canonical_dfs: guarda solo 5 filas por servicio para visualización posterior
canonical_dfs = {}
total_written  = 0

for service in ["yellow", "green", "fhvhv"]:
    success_files = (
        inventory_df
        .filter(f"read_status = 'SUCCESS' AND service_type = '{service}'")
        .select("file_path", "file_name")
        .collect()
    )

    n_rows_service = 0
    service_sample = None

    for row in success_files:
        # ── Leer con pyarrow nativo (evita el lento spark.createDataFrame) ──
        with open(row["file_path"], "rb") as fh:
            df_pd = pd.read_parquet(fh, engine="pyarrow")

        # ── Aplicar esquema canónico en pandas (10-100x más rápido) ─────────
        df_canonical = apply_canonical_schema_pd(df_pd, service, row["file_name"], config)
        del df_pd; gc.collect()

        n_rows = len(df_canonical)
        n_rows_service += n_rows

        # Guardar muestra pequeña del primer archivo para visualización
        if service_sample is None:
            service_sample = df_canonical.head(5).copy()

        # ── Escribir a bronze particionado INMEDIATAMENTE (sin acumular RAM) ─
        for (yr, mo), grp in df_canonical.groupby(["year", "month"]):
            part_dir = (
                bronze_path_dir
                / f"service_type={service}"
                / f"year={int(yr)}"
                / f"month={int(mo)}"
            )
            part_dir.mkdir(parents=True, exist_ok=True)
            out_file = part_dir / f"{row['file_name']}.parquet"
            grp.to_parquet(str(out_file), index=False, engine="pyarrow")

        del df_canonical; gc.collect()
        total_written += n_rows
        print(f"  {row['file_name']} -> {n_rows:,} registros canónicos ✓")

    if service_sample is not None:
        canonical_dfs[service] = service_sample
        print(f"\n[{service}] Total: {n_rows_service:,} registros | "
              f"Columnas: {len(service_sample.columns)}\n")
    else:
        print(f"[{service}] Sin archivos SUCCESS.\n")

print(f"Bronze escrito: {total_written:,} filas totales en {bronze_path_dir}")

Bronze output: C:\Users\HP OMEN\Modelado\SegundoParcial\Proyecto_Grupo1\data\bronze

  yellow_tripdata_2023-01.parquet -> 3,066,766 registros canónicos ✓
  yellow_tripdata_2023-02.parquet -> 2,913,955 registros canónicos ✓
  yellow_tripdata_2023-03.parquet -> 3,403,766 registros canónicos ✓
  yellow_tripdata_2023-04.parquet -> 3,288,250 registros canónicos ✓
  yellow_tripdata_2022-01.parquet -> 2,463,931 registros canónicos ✓
  yellow_tripdata_2022-02.parquet -> 2,979,431 registros canónicos ✓
  yellow_tripdata_2021-01.parquet -> 1,369,769 registros canónicos ✓
  yellow_tripdata_2020-01.parquet -> 6,405,008 registros canónicos ✓

[yellow] Total: 25,890,876 registros | Columnas: 24

  green_tripdata_2023-01.parquet -> 68,211 registros canónicos ✓
  green_tripdata_2023-02.parquet -> 64,809 registros canónicos ✓

[green] Total: 133,020 registros | Columnas: 24

  fhvhv_tripdata_2023-01.parquet -> 18,479,031 registros canónicos ✓

[fhvhv] Total: 18,479,031 registros | Columnas: 24

Bronze 

In [13]:
# Verificación: mostrar primeras 5 filas de cada servicio canónico
COLS_SHOW = [
    "trip_id", "service_type", "vendor_id",
    "pickup_datetime", "dropoff_datetime",
    "trip_distance", "fare_amount", "total_amount", "quality_status",
]

for service, df_sample in canonical_dfs.items():
    print(f"\n=== {service.upper()} — primeras 5 filas ===")
    cols = [c for c in COLS_SHOW if c in df_sample.columns]
    print(df_sample[cols].to_string(index=False))


=== YELLOW — primeras 5 filas ===
                                                         trip_id service_type  vendor_id     pickup_datetime    dropoff_datetime  trip_distance  fare_amount  total_amount quality_status
ca5b14729e0f4f4c5fb4646a409306fe8a52ef548ff7655d62e90a4c8778e789       yellow          2 2023-01-01 00:32:10 2023-01-01 00:40:36           0.97          9.3         14.30        PENDING
58d8c8bde0114d6f8b6ab7345080c18096ff6fac84ea4de95506b69343622386       yellow          2 2023-01-01 00:55:08 2023-01-01 01:01:27           1.10          7.9         16.90        PENDING
c5934b54f0b57159b4a9ae1cb1eae2218004f9189289849e72992fe0b2903289       yellow          2 2023-01-01 00:25:04 2023-01-01 00:37:49           2.51         14.9         34.90        PENDING
cd85d375ef2b10fe6dfe81b64a8f915f14f4e0380f9c8100daae4eeea7ee951c       yellow          1 2023-01-01 00:03:48 2023-01-01 00:13:25           1.90         12.1         20.85        PENDING
e239f08c0558525e3e44ea9bdc33c2a638b

### Clasificación de archivos dañados → data/quarantine/

Los archivos con `read_status` en CORRUPT, EMPTY o SCHEMA_ERROR se clasifican en una de las 7 categorías de quarantine y se genera un JSON de trazabilidad por cada uno.

In [14]:
from pathlib import Path

quarantine_dir = str(Path(config["paths"]["quarantine"]))
problem_files = (
    inventory_df
    .filter("read_status IN ('CORRUPT', 'EMPTY', 'SCHEMA_ERROR')")
    .select("file_name", "file_path", "read_status", "error_message")
    .collect()
)

quarantine_records = []
for row in problem_files:
    classification = classify_corrupt_file(row["file_path"], row["error_message"] or "")
    record = generate_quarantine_record(
        file_name=row["file_name"],
        file_path=row["file_path"],
        classification=classification,
        exception_text=row["error_message"] or f"read_status={row['read_status']}",
        phase="EXTRACT",
        quarantine_dir=quarantine_dir,
    )
    quarantine_records.append(record)
    print(f"  {row['file_name']:45s} -> {classification}")

print(f"\nTotal archivos en quarantine: {len(quarantine_records)}")
import os
print("JSONs generados en:", quarantine_dir)
print("Archivos:", os.listdir(quarantine_dir))

  PARQUET-1481.parquet                          -> RECUPERABLE_MISSING_COLUMNS
  README.md                                     -> NOT_RECOVERABLE_CORRUPT_METADATA

Total archivos en quarantine: 2
JSONs generados en: data\quarantine
Archivos: ['.gitkeep', 'PARQUET-1481.json', 'README.json']


### Escritura de data/bronze/ — particionado por service_type/year/month

Se unen los tres DataFrames canónicos y se escribe el resultado en formato Parquet particionado.

In [15]:
# Bronze ya fue escrito archivo por archivo en la celda anterior.
# Esta celda solo verifica que los archivos existen y muestra el resumen.
from pathlib import Path
import pandas as pd

bronze_path_dir = Path(str(PROJECT_ROOT / config["paths"]["bronze"]))
pq_files = sorted(bronze_path_dir.rglob("*.parquet"))

print(f"Bronze en: {bronze_path_dir}")
print(f"Archivos parquet: {len(pq_files)}")
print(f"Filas totales:    {total_written:,}")
print()
print("Particiones escritas:")
for f in pq_files[:25]:
    print(f"  {f.relative_to(bronze_path_dir)}")
if len(pq_files) > 25:
    print(f"  ... y {len(pq_files) - 25} más")

Bronze en: C:\Users\HP OMEN\Modelado\SegundoParcial\Proyecto_Grupo1\data\bronze
Archivos parquet: 68
Filas totales:    44,502,927

Particiones escritas:
  service_type=fhvhv\year=2023\month=1\fhvhv_tripdata_2023-01.parquet.parquet
  service_type=green\year=2008\month=12\green_tripdata_2023-02.parquet.parquet
  service_type=green\year=2009\month=1\green_tripdata_2023-01.parquet.parquet
  service_type=green\year=2022\month=12\green_tripdata_2023-01.parquet.parquet
  service_type=green\year=2023\month=1\green_tripdata_2023-01.parquet.parquet
  service_type=green\year=2023\month=1\green_tripdata_2023-02.parquet.parquet
  service_type=green\year=2023\month=2\green_tripdata_2023-01.parquet.parquet
  service_type=green\year=2023\month=2\green_tripdata_2023-02.parquet.parquet
  service_type=green\year=2023\month=3\green_tripdata_2023-02.parquet.parquet
  service_type=yellow\year=2001\month=1\yellow_tripdata_2023-03.parquet.parquet
  service_type=yellow\year=2001\month=1\yellow_tripdata_2023-04

In [17]:
from pathlib import Path
import pandas as pd

bronze_path = str(PROJECT_ROOT / config["paths"]["bronze"])
pq_files = sorted(Path(bronze_path).rglob("*.parquet"))
print(f"Archivos parquet en bronze: {len(pq_files)}")

rows_by_svc = {}
col_count = None
col_names = []
for pq_file in pq_files:
    with open(str(pq_file), "rb") as _fh:
        _df = pd.read_parquet(_fh, engine="pyarrow")
    svc_part = next((p for p in pq_file.parts if p.startswith("service_type=")), None)
    svc = svc_part.split("=")[1] if svc_part else pq_file.parent.parent.parent.name
    rows_by_svc[svc] = rows_by_svc.get(svc, 0) + len(_df)
    if col_count is None:
        col_count = len(_df.columns)
        col_names = list(_df.columns)
    del _df

print(f"Columnas: {col_count} (debe ser 24)")
print(f"Campos: {col_names}")
print()
for svc, cnt in sorted(rows_by_svc.items()):
    print(f"  {svc}: {cnt:,} registros")

Archivos parquet en bronze: 68
Columnas: 24 (debe ser 24)
Campos: ['trip_id', 'service_type', 'vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'pickup_location_id', 'dropoff_location_id', 'payment_type', 'fare_amount', 'extra_amount', 'mta_tax', 'tip_amount', 'tolls_amount', 'total_amount', 'congestion_surcharge', 'airport_fee', 'year', 'month', 'source_file', 'ingestion_timestamp', 'improvement_surcharge', 'quality_status']

  fhvhv: 18,479,031 registros
  green: 133,020 registros
  yellow: 25,890,876 registros
